# Lab 3: LSTM Forecasting of Monthly Precipitation (Hawaii, 1939 to 2024)

Forecasts next month precipitation from the previous 12 months with a stacked LSTM in
PyTorch, using a chronological train/test split so the model never sees the future.

**Run in Google Colab.** Upload `HW_precipitation_1939_2024.csv` (columns `Date` as YYYYMM and
`Precipitation` in mm).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DATA_PATH = "HW_precipitation_1939_2024.csv"
WINDOW = 12

In [ ]:
data = pd.read_csv(DATA_PATH)
data["Date"] = pd.to_datetime(data["Date"].astype(str), format="%Y%m")
data = data.sort_values("Date").reset_index(drop=True)

plt.figure(figsize=(12, 4))
plt.plot(data["Date"], data["Precipitation"])
plt.title("Monthly precipitation in Hawaii"); plt.xlabel("Date"); plt.ylabel("Precipitation (mm)")
plt.tight_layout(); plt.show()

## Build sequences: 12 months in, next month out

In [ ]:
values = data["Precipitation"].values.astype("float32").reshape(-1, 1)

split = int(len(values) * 0.8)                 # chronological split
scaler = MinMaxScaler().fit(values[:split])    # fit on the training period only
scaled = scaler.transform(values).astype("float32")

def make_sequences(series, window):
    X, y = [], []
    for i in range(len(series) - window):
        X.append(series[i:i + window])
        y.append(series[i + window])
    return np.array(X), np.array(y)

X, y = make_sequences(scaled, WINDOW)          # X: (n, 12, 1), y: (n, 1)
n_train = split - WINDOW
X_train, y_train = X[:n_train], y[:n_train]
X_test, y_test = X[n_train:], y[n_train:]
test_dates = data["Date"].values[WINDOW + n_train:]
print(X_train.shape, X_test.shape)

train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(torch.tensor(X_test), torch.tensor(y_test)), batch_size=64, shuffle=False)

## Model

In [ ]:
class TimeSeriesLSTM(nn.Module):
    def __init__(self, hidden=40, layers=3):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden, num_layers=layers, batch_first=True)
        self.linear = nn.Linear(hidden, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.linear(out[:, -1, :])      # prediction from the last time step

model = TimeSeriesLSTM()
criterion = nn.L1Loss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
history = []
for epoch in range(100):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        test_loss = np.mean([criterion(model(xb), yb).item() for xb, yb in test_loader])
    history.append(test_loss)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}: test MAE (scaled) {test_loss:.4f}")

## Evaluate in millimetres

In [ ]:
model.eval()
with torch.no_grad():
    preds_scaled = model(torch.tensor(X_test)).numpy()
preds = scaler.inverse_transform(preds_scaled).ravel()
truth = scaler.inverse_transform(y_test).ravel()
mae_mm = np.mean(np.abs(preds - truth))
naive_mae = np.mean(np.abs(truth[1:] - truth[:-1]))   # baseline: next month equals this month
print(f"LSTM test MAE: {mae_mm:.1f} mm | naive baseline MAE: {naive_mae:.1f} mm")

plt.figure(figsize=(12, 4))
plt.plot(test_dates, truth, label="Observed")
plt.plot(test_dates, preds, label="LSTM forecast", alpha=0.8)
plt.title("Test period: observed vs forecast"); plt.ylabel("Precipitation (mm)")
plt.legend(); plt.tight_layout(); plt.show()